# Aula 04: Pré-processamento: Escalonamento, Encoding e Feature Engineering


## Objetivos de aprendizagem

Ao final desta aula você será capaz de:

1. **Escalonar** variáveis numéricas (normalização e padronização) e escolher o método adequado.
2. **Codificar** variáveis categóricas (one-hot, ordinal, label, target encoding).
3. **Montar pipelines** com `ColumnTransformer` e `Pipeline` para evitar *data leakage*.
4. **Criar novas variáveis** (*feature engineering*) que aumentam o poder preditivo dos modelos.

---

## Por que pré-processar?

Depois de **limpar** os dados, ainda precisamos **transformá-los** para que os algoritmos funcionem bem. Dois motivos centrais:

- **Escala das variáveis:** modelos baseados em distância (KNN, SVM, k-means) e em gradiente (regressão logística, redes neurais) são sensíveis à magnitude. Uma coluna em "reais" (0–100000) domina outra em "anos" (0–80) se não forem colocadas na mesma escala.
- **Tipos das variáveis:** a maioria dos algoritmos só aceita **números**. Variáveis categóricas ("cidade", "sexo") precisam ser convertidas para uma representação numérica adequada.

> Todo parâmetro de transformação (média, desvio, mínimo, categorias) deve ser aprendido **apenas no conjunto de treino** e depois aplicado ao teste. Faremos isso com `Pipeline`.

## 1. Escalonamento de variáveis numéricas

Colocar as variáveis numa escala comparável. Os três métodos mais usados:

| Método | Fórmula | Resultado | Quando usar |
|---|---|---|---|
| **Normalização (Min-Max)** | \((x - x_{min}) / (x_{max} - x_{min})\) | valores em \[0, 1\] | dados limitados; redes neurais |
| **Padronização (Z-score)** | \((x - \mu) / \sigma\) | média 0, desvio 1 | padrão geral; dados ~normais |
| **Robust Scaler** | \((x - mediana) / IQR\) | robusto a outliers | dados com muitos outliers |

- **Normalização vs. Padronização:** normalização "espreme" tudo num intervalo fixo (bom quando há limites naturais); padronização centra em 0 e é menos sensível a extremos.
- **`RobustScaler`** usa mediana e IQR, por isso quase não é afetado por outliers, útil quando a limpeza (Aula 05) não removeu todos.

A seguir, comparamos os três lado a lado em uma mesma variável.


In [ ]:
# Comparando MinMax, Standard e Robust em uma variável real
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

tips = sns.load_dataset('tips')
x = tips[['total_bill']]  # scalers esperam formato 2D (coluna)

comparacao = pd.DataFrame({
    'original':  x['total_bill'],
    'minmax':    MinMaxScaler().fit_transform(x).ravel(),
    'standard':  StandardScaler().fit_transform(x).ravel(),
    'robust':    RobustScaler().fit_transform(x).ravel(),
})
print(comparacao.describe().round(2))

# Histogramas: a FORMA da distribuição não muda, apenas a ESCALA do eixo
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, col in zip(axes, comparacao.columns):
    ax.hist(comparacao[col], bins=25)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 2. Codificação de variáveis categóricas (*encoding*)

Algoritmos precisam de números. Como transformar "sexo", "cidade" ou "escolaridade" em números **sem inventar relações falsas**?

| Técnica | Como funciona | Quando usar | Cuidado |
|---|---|---|---|
| **One-Hot** | uma coluna binária por categoria | categóricas **nominais** (sem ordem) | explode dimensão se houver muitas categorias |
| **Ordinal** | mapeia categorias para inteiros **com ordem** | categóricas **ordinais** (ex.: baixo < médio < alto) | não usar em nominais! |
| **Label** | inteiros arbitrários (0,1,2...) | apenas para o **alvo** (`y`), não para features | cria ordem falsa se usado em X |
| **Target/Mean** | substitui a categoria pela média do alvo | alta cardinalidade | risco de *leakage*; use com validação |

**Erro comum:** aplicar *Label Encoding* em uma variável nominal (ex.: `cidade → 0,1,2`). O modelo passa a achar que "cidade 2 > cidade 1", o que não faz sentido. Para nominais, use **One-Hot**.


In [ ]:
# Demonstração de One-Hot vs. Ordinal em colunas do dataset 'tips'
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

amostra = tips[['day', 'size']].head(6).copy()
print('Original:')
print(amostra)

In [ ]:
# One-Hot em 'day' (nominal): cria uma coluna 0/1 por dia
ohe = OneHotEncoder(sparse_output=False)
dummies = ohe.fit_transform(tips[['day']])
df_ohe = pd.DataFrame(dummies, columns=ohe.get_feature_names_out(['day']))
print('\nOne-Hot de "day" (primeiras linhas):')
print(df_ohe.head(6))

In [ ]:
# Alternativa rápida com pandas (equivalente ao One-Hot):
print('\npd.get_dummies("day"):')
print(pd.get_dummies(tips['day'], prefix='day').head(3))

In [ ]:
# Ordinal em uma variável COM ordem (exemplo didático de escolaridade)
ordem = [['fundamental', 'medio', 'superior']]
oe = OrdinalEncoder(categories=ordem)
exemplo = pd.DataFrame({'escolaridade': ['medio', 'superior', 'fundamental', 'medio']})
exemplo['codigo'] = oe.fit_transform(exemplo[['escolaridade']]).astype(int)
print('\nOrdinal (respeita a ordem fundamental<medio<superior):')
print(exemplo)

## 3. Pipelines e `ColumnTransformer`

Fazer imputação + escalonamento + encoding "na mão" é frágil e propenso a **data leakage** (vazamento de informação do teste para o treino). A solução do scikit-learn:

- **`Pipeline`**: encadeia passos (ex.: imputar → escalonar) numa única sequência.
- **`ColumnTransformer`**: aplica pipelines **diferentes** a grupos de colunas diferentes (numéricas × categóricas) ao mesmo tempo.

**Vantagens:**

1. **Sem *leakage*:** `fit` aprende os parâmetros só no treino; `transform` os reaplica ao teste.
2. **Reprodutível:** todo o pré-processamento vira um único objeto, fácil de salvar e reusar.
3. **Compatível com validação cruzada e busca de hiperparâmetros.**

O exemplo abaixo usa o dataset **Titanic** (já disponível no seaborn, sem download): numéricas recebem imputação por mediana + padronização; categóricas recebem imputação por moda + one-hot.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import pandas as pd
import seaborn as sns

# Titanic (já vem no seaborn): numéricas + categóricas + missing.
df = sns.load_dataset("titanic")

# Separar as colunas por tipo: cada grupo terá um tratamento diferente
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
print("Numéricas :", num_cols)
print("Categóricas:", cat_cols)
print("Shape:", df.shape)

In [ ]:
# Pipeline das NUMÉRICAS: imputa ausentes pela mediana, depois padroniza (z-score)
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

In [ ]:
# Pipeline das CATEGÓRICAS: imputa pela moda, depois aplica one-hot
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

In [ ]:
# ColumnTransformer: aplica cada pipeline ao seu grupo de colunas
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols),
])

In [ ]:
# fit_transform aprende os parâmetros e transforma de uma vez.
# Num fluxo real: fit_transform no TREINO e apenas transform no TESTE.
X = preprocessor.fit_transform(df)
print('Shape original :', df.shape)
print('Shape processado:', X.shape, '(colunas aumentaram por causa do one-hot)')

## 4. *Feature engineering*: criando variáveis melhores

*Feature engineering* é a arte de criar novas variáveis a partir das existentes para dar ao modelo informação mais útil. Muitas vezes vale mais que trocar de algoritmo.

**Técnicas comuns:**

| Técnica | Exemplo |
|---|---|
| **Combinações aritméticas** | `densidade = massa / volume`; `preco_por_m2 = preco / area` |
| **Razões e proporções** | `acidez_total = acidez_fixa + acidez_volatil` |
| **Binning (discretização)** | transformar idade em faixas: criança/adulto/idoso (`pd.cut`) |
| **Extração de datas** | de uma data: dia da semana, mês, é fim de semana? |
| **Interações** | produto de duas variáveis que juntas importam |
| **Transformações** | `log`, `sqrt` para reduzir assimetria |

> Boas *features* incorporam **conhecimento do domínio**. Pergunte a um especialista: "que combinações dessas medidas fazem sentido no seu campo?"

No exercício a seguir você criará *features* derivadas para o dataset **Wine Quality**.


### Dataset de apoio — Wine Quality (vinho tinto)

Cada linha é uma amostra de vinho com medidas físico-químicas (acidez, açúcar, álcool...) e uma nota de qualidade (`quality`).

In [ ]:
from pathlib import Path
destino = Path.cwd() / "dados" / "winequality-red.csv"
destino.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
import urllib.request
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

In [ ]:
urllib.request.urlretrieve(url, destino)

In [ ]:
wine = pd.read_csv(destino, sep=";")  # o CSV da UCI usa ';'
print("Shape:", wine.shape)
print("Colunas:", list(wine.columns))

In [ ]:
# Features derivadas (exemplos prontos — o exercício pede que você crie as suas)
wine["acidez_total"] = wine["fixed acidity"] + wine["volatile acidity"]
wine["razao_sulfitos"] = wine["free sulfur dioxide"] / wine["total sulfur dioxide"]
wine["alcool_por_acidez"] = wine["alcohol"] / wine["fixed acidity"]
wine["bom"] = (wine["quality"] >= 7).astype(int)
print("Novas colunas:", ["acidez_total", "razao_sulfitos", "alcool_por_acidez", "bom"])
wine[["quality", "acidez_total", "razao_sulfitos", "alcool_por_acidez", "bom"]].head()

## Exercício: Pré-processamento de ponta a ponta

Usando o dataset **Wine Quality** (carregado acima):

1. **Feature engineering:** crie ao menos 3 variáveis derivadas com sentido químico, por exemplo:
   - `acidez_total = fixed acidity + volatile acidity`
   - `razao_sulfitos = free sulfur dioxide / total sulfur dioxide`
   - `alcool_por_acidez = alcohol / fixed acidity`
2. **Alvo binário:** crie `bom = (quality >= 7).astype(int)`.
3. **Split treino/teste:** use `train_test_split` com `stratify=y` e `random_state=42`.
4. **Pipeline:** monte um `Pipeline` com `StandardScaler`.